In [ ]:
# ============================================================
# HER2 RFantibody Pipeline
#
# Goal:
#   1. Download HER2/trastuzumab complex PDB: 1N8Z
#   2. Extract HER2 chain A as target chain T
#   3. Copy RFantibody scFv framework
#   4. Identify HER2 antibody-contact hotspot residues
#   5. Run RFantibody RFdiffusion
#   6. Run ProteinMPNN sequence design
# ============================================================

# -----------------------------
# 0. Mount Drive + paths
# -----------------------------
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/her2_project")
RFAB_DIR = PROJECT_DIR / "rfantibody_her2"

INPUT_DIR = RFAB_DIR / "inputs"
OUTPUT_DIR = RFAB_DIR / "outputs"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HER2_ORIG_PDB = INPUT_DIR / "her2_1n8z.pdb"
HER2_TARGET_PDB = INPUT_DIR / "her2_target.pdb"
FRAMEWORK_PDB = INPUT_DIR / "hu-4D5-8_Fv.pdb"

print("Project:", RFAB_DIR)

Mounted at /content/drive
Project: /content/drive/MyDrive/her2_project/rfantibody_her2


In [ ]:
# -----------------------------
# 1. Download HER2 complex PDB
# -----------------------------
!wget -q https://files.rcsb.org/download/1N8Z.pdb -O "{HER2_ORIG_PDB}"

print("Downloaded:", HER2_ORIG_PDB)

Downloaded: /content/drive/MyDrive/her2_project/rfantibody_her2/inputs/her2_1n8z.pdb


In [ ]:
# -----------------------------
# 2. Install dependencies
# -----------------------------
!pip install -q biopython

from Bio.PDB import PDBParser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.1 MB/s eta 0:00:00


In [ ]:
# -----------------------------
# 3. Extract HER2 chain A -> rename as chain T
#
# RFantibody target chain is usually treated as T.
# 1N8Z:
#   Chain A = HER2
#   Chain B/C = trastuzumab antibody chains
# -----------------------------
with open(HER2_ORIG_PDB) as f_in, open(HER2_TARGET_PDB, "w") as f_out:
    for line in f_in:
        if line.startswith("ATOM") and line[21] == "A":
            line = line[:21] + "T" + line[22:]
            f_out.write(line)
    f_out.write("END\n")

print("Saved HER2 target:", HER2_TARGET_PDB)

Saved HER2 target: /content/drive/MyDrive/her2_project/rfantibody_her2/inputs/her2_target.pdb


In [ ]:
# -----------------------------
# 4. Sanity-check extracted HER2 target
# -----------------------------
parser = PDBParser(QUIET=True)
structure = parser.get_structure("her2_target", HER2_TARGET_PDB)
model = list(structure.get_models())[0]

chains = list(model.get_chains())
print("Chains:", [c.id for c in chains])

for chain in chains:
    residues = [r for r in chain if r.id[0] == " "]
    resnums = [r.id[1] for r in residues]
    print(
        f"Chain {chain.id}:",
        f"{len(resnums)} residues",
        f"range {min(resnums)}-{max(resnums)}"
    )

!grep -c "^ATOM" "{HER2_TARGET_PDB}"
!head -n 5 "{HER2_TARGET_PDB}"

Chains: ['T']
Chain T: 214 residues range 1-214
1645
ATOM      1  N   ASP T   1      34.539  88.752  88.298  1.00 30.87           N  
ATOM      2  CA  ASP T   1      34.791  87.461  89.035  1.00 30.75           C  
ATOM      3  C   ASP T   1      35.907  86.779  88.254  1.00 29.03           C  
ATOM      4  O   ASP T   1      35.694  86.356  87.114  1.00 29.42           O  
ATOM      5  CB  ASP T   1      33.536  86.583  89.061  1.00 30.97           C  


In [ ]:
%%bash
# -----------------------------
# 5. Install RFantibody
# -----------------------------
cd /content

if [ ! -d RFantibody ]; then
  git clone https://github.com/RosettaCommons/RFantibody.git
fi

cd RFantibody

bash include/download_weights.sh

curl -LsSf https://astral.sh/uv/install.sh | sh
source ~/.bashrc

uv sync

All weights downloaded successfully!
installing to /usr/local/bin
  uv
  uvx
everything's installed!


Cloning into 'RFantibody'...
--2026-05-04 15:44:12--  https://files.ipd.uw.edu/pub/RFantibody/RFdiffusion_Ab.pt
Resolving files.ipd.uw.edu (files.ipd.uw.edu)... 128.95.160.134, 128.95.160.135, 2607:4000:406::160:135, ...
Connecting to files.ipd.uw.edu (files.ipd.uw.edu)|128.95.160.134|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 483452922 (461M) [application/octet-stream]
Saving to: ‘RFdiffusion_Ab.pt’

     0K .......... .......... .......... .......... ..........  0% 1.18M 6m31s
    50K .......... .......... .......... .......... ..........  0% 1.18M 6m30s
   100K .......... .......... .......... .......... ..........  0%  193M 4m21s
   150K .......... .......... .......... .......... ..........  0%  192M 3m16s
   200K .......... .......... .......... .......... ..........  0% 1.19M 3m54s
   250K .......... .......... .......... .......... ..........  0%  200M 3m16s
   300K .......... .......... .......... .......... ..........  0%  363M 2m48s
   350K ...

In [ ]:
# -----------------------------
# 6. Copy RFantibody example scFv framework
# -----------------------------
!cp /content/RFantibody/scripts/examples/example_inputs/hu-4D5-8_Fv.pdb "{FRAMEWORK_PDB}"

print("Framework:", FRAMEWORK_PDB)
!ls -lh "{INPUT_DIR}"

Framework: /content/drive/MyDrive/her2_project/rfantibody_her2/inputs/hu-4D5-8_Fv.pdb
total 947K
-rw------- 1 root root 688K May  4 15:43 her2_1n8z.pdb
-rw------- 1 root root 131K May  4 15:43 her2_target.pdb
-rw------- 1 root root 129K May  4 15:46 hu-4D5-8_Fv.pdb


In [ ]:
# -----------------------------
# 7. Identify HER2 residues contacting original antibody
#
# This helps choose hotspot residues for RFdiffusion.
# We find HER2 residues within 4.5 Å of antibody chains B/C.
# -----------------------------
from Bio.PDB import NeighborSearch

orig = parser.get_structure("orig_1n8z", HER2_ORIG_PDB)
target = parser.get_structure("target", HER2_TARGET_PDB)

orig_model = list(orig.get_models())[0]
target_model = list(target.get_models())[0]

HER2_CHAIN = "A"
AB_CHAINS = ["B", "C"]

ab_atoms = []
for chain_id in AB_CHAINS:
    ab_atoms.extend([
        atom for atom in orig_model[chain_id].get_atoms()
        if atom.element != "H"
    ])

ns = NeighborSearch(ab_atoms)

contact_residues = []

for residue in orig_model[HER2_CHAIN]:
    if residue.id[0] != " ":
        continue

    for atom in residue:
        if atom.element != "H" and ns.search(atom.coord, 4.5, level="A"):
            contact_residues.append(residue.id[1])
            break

print("Original HER2 contact residues:")
print(contact_residues)

Original HER2 contact residues:
[28, 30, 31, 32, 34, 36, 38, 42, 43, 44, 46, 49, 50, 53, 55, 66, 87, 89, 91, 92, 93, 94, 95, 96, 98, 100, 114, 116, 117, 118, 121, 123, 124, 129, 131, 133, 135, 137, 138, 160, 161, 162, 163, 164, 174, 175, 176, 180, 207, 208, 209]


In [ ]:
# -----------------------------
# 8. Map original HER2 chain A residue numbers
#    to extracted target chain T residue numbers
# -----------------------------
orig_residues = [r for r in orig_model["A"] if r.id[0] == " "]
target_residues = [r for r in target_model["T"] if r.id[0] == " "]

residue_map = {
    orig_r.id[1]: target_r.id[1]
    for orig_r, target_r in zip(orig_residues, target_residues)
}

mapped_hotspots = [
    residue_map[r]
    for r in contact_residues
    if r in residue_map
]

hotspot_string = ",".join([f"T{x}" for x in mapped_hotspots[:5]])

print("Mapped RFantibody hotspots:")
print(mapped_hotspots)

print("\nSuggested hotspot string:")
print(hotspot_string)

Mapped RFantibody hotspots:
[28, 30, 31, 32, 34, 36, 38, 42, 43, 44, 46, 49, 50, 53, 55, 66, 87, 89, 91, 92, 93, 94, 95, 96, 98, 100, 114, 116, 117, 118, 121, 123, 124, 129, 131, 133, 135, 137, 138, 160, 161, 162, 163, 164, 174, 175, 176, 180, 207, 208, 209]

Suggested hotspot string:
T28,T30,T31,T32,T34


In [ ]:
# -----------------------------
# 9. Run RFantibody RFdiffusion
#
# Inputs:
#   -t target HER2 chain T
#   -f antibody framework
#   -q output quiver file
#   -n number of backbone designs
#   -l CDR length ranges
#   -h hotspot residues on target
# -----------------------------
%cd /content/RFantibody

RFDIFF_QV = OUTPUT_DIR / "1_rfdiffusion_hotspot_test.qv"
RFDIFF_LOG = OUTPUT_DIR / "rfdiffusion.log"

!uv run rfdiffusion \
  -t "{HER2_TARGET_PDB}" \
  -f "{FRAMEWORK_PDB}" \
  -q "{RFDIFF_QV}" \
  -n 1 \
  -l "H1:7,H2:6,H3:5-13,L1:8-13,L2:7,L3:9-11" \
  -h "T30,T31,T32" \
  --deterministic \
  > "{RFDIFF_LOG}" 2>&1

print("RFdiffusion output:", RFDIFF_QV)
print("RFdiffusion log:", RFDIFF_LOG)

/content/RFantibody
RFdiffusion output: /content/drive/MyDrive/her2_project/rfantibody_her2/outputs/1_rfdiffusion_hotspot_test.qv
RFdiffusion log: /content/drive/MyDrive/her2_project/rfantibody_her2/outputs/rfdiffusion.log


In [ ]:
# -----------------------------
# 10. Check RFdiffusion output
# -----------------------------
!ls -lh "{OUTPUT_DIR}"
!tail -n 40 "{RFDIFF_LOG}"

total 202M
-rw------- 1 root root  72M May  4 02:09 1_rfdiffusion_hotspot_pX0_traj.qv
-rw------- 1 root root 814K May  4 02:08 1_rfdiffusion_hotspot.qv
-rw------- 1 root root  11M May  4 15:57 1_rfdiffusion_hotspot_test_pX0_traj.qv
-rw------- 1 root root 116K May  4 15:56 1_rfdiffusion_hotspot_test.qv
-rw------- 1 root root 4.3M May  4 15:57 1_rfdiffusion_hotspot_test_Xt-1_traj.qv
-rw------- 1 root root  31M May  4 02:09 1_rfdiffusion_hotspot_Xt-1_traj.qv
-rw------- 1 root root  51M May  4 00:27 1_rfdiffusion_pX0_traj.qv
-rw------- 1 root root 581K May  4 00:27 1_rfdiffusion.qv
-rw------- 1 root root  22M May  4 00:27 1_rfdiffusion_Xt-1_traj.qv
-rw------- 1 root root 6.4M May  4 02:52 2_proteinmpnn_hotspot.qv
-rw------- 1 root root 2.3M May  4 00:28 2_proteinmpnn.qv
-rw------- 1 root root 3.0M May  4 00:45 3_rf2.qv
-rw------- 1 root root 1003 May  4 00:48 3_rf2.sc
-rw------- 1 root root   87 May  4 00:48 3_rf2_scores.tsv
drwx------ 2 root root 4.0K May  4 03:09 chai_validation
drwx----

In [ ]:
# -----------------------------
# 11. Run ProteinMPNN sequence design
#
# Inputs:
#   -q RFdiffusion quiver output
#   --output-quiver designed sequence quiver
#   -n number of sequences per backbone
#   -t sampling temperature
# -----------------------------
MPNN_QV = OUTPUT_DIR / "2_proteinmpnn_hotspot_test.qv"
MPNN_LOG = OUTPUT_DIR / "proteinmpnn.log"

!uv run proteinmpnn \
  -q "{RFDIFF_QV}" \
  --output-quiver "{MPNN_QV}" \
  -n 8 \
  -t 0.2 \
  > "{MPNN_LOG}" 2>&1

print("ProteinMPNN output:", MPNN_QV)
print("ProteinMPNN log:", MPNN_LOG)

ProteinMPNN output: /content/drive/MyDrive/her2_project/rfantibody_her2/outputs/2_proteinmpnn_hotspot_test.qv
ProteinMPNN log: /content/drive/MyDrive/her2_project/rfantibody_her2/outputs/proteinmpnn.log


In [ ]:
# -----------------------------
# 12. Check ProteinMPNN output
# -----------------------------
!ls -lh "{OUTPUT_DIR}"
!tail -n 40 "{MPNN_LOG}"

total 203M
-rw------- 1 root root  72M May  4 02:09 1_rfdiffusion_hotspot_pX0_traj.qv
-rw------- 1 root root 814K May  4 02:08 1_rfdiffusion_hotspot.qv
-rw------- 1 root root  11M May  4 15:57 1_rfdiffusion_hotspot_test_pX0_traj.qv
-rw------- 1 root root 116K May  4 15:56 1_rfdiffusion_hotspot_test.qv
-rw------- 1 root root 4.3M May  4 15:57 1_rfdiffusion_hotspot_test_Xt-1_traj.qv
-rw------- 1 root root  31M May  4 02:09 1_rfdiffusion_hotspot_Xt-1_traj.qv
-rw------- 1 root root  51M May  4 00:27 1_rfdiffusion_pX0_traj.qv
-rw------- 1 root root 581K May  4 00:27 1_rfdiffusion.qv
-rw------- 1 root root  22M May  4 00:27 1_rfdiffusion_Xt-1_traj.qv
-rw------- 1 root root 6.4M May  4 02:52 2_proteinmpnn_hotspot.qv
-rw------- 1 root root 922K May  4 16:10 2_proteinmpnn_hotspot_test.qv
-rw------- 1 root root 2.3M May  4 00:28 2_proteinmpnn.qv
-rw------- 1 root root 3.0M May  4 00:45 3_rf2.qv
-rw------- 1 root root 1003 May  4 00:48 3_rf2.sc
-rw------- 1 root root   87 May  4 00:48 3_rf2_score

In [ ]:
# -----------------------------
# 13. Next expected steps
#
# After this:
#   1. Run RF2 / structure prediction if RFantibody provides the command
#   2. Extract designed PDBs / FASTA sequences
#   3. Score candidates with:
#        - RFantibody internal scores
#        - ipTM / interface confidence
#        - ESM2 surrogate score
#        - developability filters
# -----------------------------
print("Pipeline complete through RFdiffusion + ProteinMPNN.")
print("Main output:", MPNN_QV)

Pipeline complete through RFdiffusion + ProteinMPNN.
Main output: /content/drive/MyDrive/her2_project/rfantibody_her2/outputs/2_proteinmpnn_hotspot_test.qv


In [ ]:
%cd /content/RFantibody

# inspect qv
!/content/RFantibody/.venv/bin/qvls \
  /content/drive/MyDrive/her2_project/rfantibody_her2/outputs/2_proteinmpnn_hotspot.qv

# make extraction folder
!mkdir -p /content/drive/MyDrive/her2_project/rfantibody_her2/outputs/extracted

# extract inside target folder
%cd /content/drive/MyDrive/her2_project/rfantibody_her2/outputs/extracted

!/content/RFantibody/.venv/bin/qvextract \
  ../2_proteinmpnn_hotspot.qv

# check result
!ls -lh

/content/RFantibody
samples_design_0_dldesign_0
samples_design_0_dldesign_1
samples_design_0_dldesign_2
samples_design_0_dldesign_3
samples_design_0_dldesign_4
samples_design_0_dldesign_5
samples_design_0_dldesign_6
samples_design_0_dldesign_7
samples_design_1_dldesign_0
samples_design_1_dldesign_1
samples_design_1_dldesign_2
samples_design_1_dldesign_3
samples_design_1_dldesign_4
samples_design_1_dldesign_5
samples_design_1_dldesign_6
samples_design_1_dldesign_7
samples_design_2_dldesign_0
samples_design_2_dldesign_1
samples_design_2_dldesign_2
samples_design_2_dldesign_3
samples_design_2_dldesign_4
samples_design_2_dldesign_5
samples_design_2_dldesign_6
samples_design_2_dldesign_7
samples_design_3_dldesign_0
samples_design_3_dldesign_1
samples_design_3_dldesign_2
samples_design_3_dldesign_3
samples_design_3_dldesign_4
samples_design_3_dldesign_5
samples_design_3_dldesign_6
samples_design_3_dldesign_7
samples_design_4_dldesign_0
samples_design_4_dldesign_1
samples_design_4_dldesign_2


In [ ]:
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

pdb_path = "samples_design_6_dldesign_7.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("design7", pdb_path)
model = list(structure.get_models())[0]

for chain in model:
    seq = []
    for r in chain:
        if r.id[0] == " ":
            try:
                seq.append(seq1(r.resname))
            except:
                seq.append("X")

    print("\nChain:", chain.id)
    print("Length:", len(seq))
    print("Start:", "".join(seq[:20]))


Chain: H
Length: 118
Start: EVQLVESGGGLVQPGGSLRL

Chain: L
Length: 104
Start: DIQMTQSPSSLSASVGDRVT

Chain: T
Length: 214
Start: DIQMTQSPSSLSASVGDRVT
